In [ ]:
# !pip --version

pip 26.2.1 from D:\kbh\ex0918\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables.utils import ConfigurableFieldSpec
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db"
)

In [4]:
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내 이름은 테디야, 나는 랭체인 개발자야. 앞으로 잘부탁해!"
)

chat_message_history.add_ai_message("안녕 테디, 만나서 반가워. 나도 잘 부탁해!")

In [5]:
chat_message_history.messages

[HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야, 나는 랭체인 개발자야. 앞으로 잘부탁해!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [8]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

chain = prompt | ChatOpenAI(model="gpt-4o") | StrOutputParser()

In [9]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name=user_id,
        session_id=conversation_id,
        connection="sqlite:///sqlite.db"
    )

In [11]:
config_fields = [
    ConfigurableFieldSpec(
        id="user_id",
        annotation=str,
        name="User ID",
        description="Unique identifier for a user.",
        default="",
        is_shared=True,
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation.",
        default="",
        is_shared=True,
    ),
]

In [12]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="chat_history",
    history_factory_config=config_fields,
)

d:\kbh\ex0918\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation1"}}

In [14]:
chain_with_history.invoke({"question": "안녕? 반가워, 내 이름은 테디야"}, config)

'안녕하세요, 테디! 반가워요. 저는 당신을 도와드리기 위한 인공지능 어시스턴트입니다. 어떻게 도와드릴까요?'

In [15]:
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'당신의 이름은 테디라고 말씀하셨습니다. 맞나요?'

In [16]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation2"}}

chain_with_history.invoke({"question": "내이름이 뭐라고?"}, config)

'죄송하지만, 제가 사용자의 이름을 알 수 있는 방법은 없습니다. 어떻게 도와드릴까요?'